# ETA Model Training (GPU + LGBM + FT-Transformer)

Train three models for ensemble:
1. **NN** (MLP with zone embeddings) on full 37M rows, GPU
2. **LightGBM** on 10M rows, CPU
3. **FT-Transformer** on 10M rows, GPU

Works on both Google Colab and Kaggle.

In [ ]:
# 1. Install dependencies (torch is pre-installed on Colab/Kaggle)
!pip install -q huggingface_hub pyarrow tqdm mlflow lightgbm

In [ ]:
# 2. Clone repo
!git clone https://github.com/sarthakbiswas97/eta-engine.git
%cd eta-engine

In [ ]:
# 3. Download data + compute zone-pair stats
!pip install -q -r requirements.txt
!python data/download_data.py
!PYTHONPATH=. python -m features.zone_pair_stats

In [ ]:
# 4. Verify GPU and data
import torch
import os

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, "total_memory", getattr(props, "total_mem", 0)) / 1e9
    print(f"GPU: {gpu}")
    print(f"VRAM: {vram:.1f} GB")

print()
for f in ["data/train.parquet", "data/dev.parquet", "data/zone_pair_stats/zone_pair_stats.pkl"]:
    size = os.path.getsize(f) / 1e6 if os.path.exists(f) else 0
    status = f"{size:.1f} MB" if size > 0 else "MISSING"
    print(f"  {f}: {status}")

---
## Download pre-trained models from HF (skip retraining)

In [ ]:
# 4b. Download pre-trained NN + LGBM from HF Hub
# Skip this cell if you want to retrain from scratch (use cells 5a, 5b instead)
from huggingface_hub import hf_hub_download
import shutil

for filename in ["model.pt", "lgbm_model.txt"]:
    if os.path.exists(filename):
        print(f"Already exists: {filename}")
        continue
    path = hf_hub_download("sarthakbiswas/eta-engine", filename)
    shutil.copy2(path, filename)
    print(f"Downloaded {filename} ({os.path.getsize(filename)/1e6:.1f} MB)")

---
## Model 1: Neural Net (MLP)
Skip if downloaded from HF above.

In [ ]:
# 5a. Train NN (GPU, ~90 min on 37M rows) -- skip if using HF model
# !PYTHONPATH=. python train.py --epochs 10 --batch-size 8192 --lr 5e-4 --patience 3 --num-workers 2 --dev-sample 50000 --save-every 2 --loss huber --run-name v4b-huber

---
## Model 2: LightGBM
Skip if downloaded from HF above.

In [ ]:
# 5b. Train LightGBM (CPU, ~5 min on 10M rows) -- skip if using HF model
# !PYTHONPATH=. python scripts/train_lgbm.py --sample 10000000 --dev-sample 100000 --run-name lgbm-v1

---
## Model 3: FT-Transformer

In [ ]:
# 5c. Train FT-Transformer (GPU, ~30-60 min on 10M rows)
# Experiment 1: L1 loss (matches eval metric)
!PYTHONPATH=. python scripts/train_ft.py --sample 10000000 --epochs 10 --batch-size 1024 --lr 1e-4 --patience 3 --loss l1 --run-name ft-v1-l1

# Experiment 2: MSE loss (paper default)
# !PYTHONPATH=. python scripts/train_ft.py --sample 10000000 --epochs 10 --batch-size 1024 --lr 1e-4 --patience 3 --loss mse --run-name ft-v1-mse

---
## Ensemble

In [ ]:
# 5d. Find optimal 2-model ensemble weight (NN + LGBM)
!PYTHONPATH=. python scripts/find_ensemble_weight.py --dev-sample 0

In [ ]:
# 5e. Find optimal 3-model ensemble weights (NN + LGBM + FT)
# Run after FT-Transformer training completes
import numpy as np
import lightgbm as lgb
from features.pipeline import FeaturePipeline
from model.architecture import ETAModel, ModelConfig
from model.ft_transformer import FTTransformer, FTConfig
from model.dataset import create_dataloader
from scripts.train_lgbm import build_lgbm_features, load_features
from pathlib import Path
import gc

ROOT = Path(".")
pipeline = FeaturePipeline.from_artifacts(ROOT / "data" / "zone_pair_stats" / "zone_pair_stats.pkl")

# Load full dev
dev_cat, dev_cont, dev_targets = load_features(pipeline, ROOT / "data" / "dev.parquet")

# --- NN predictions ---
nn_cp = torch.load("model.pt", map_location="cpu", weights_only=False)
nn_config = ModelConfig(**nn_cp["model_config"])
nn_model = ETAModel(nn_config)
nn_model.load_state_dict(nn_cp["model_state_dict"])
nn_model.eval()
nn_norm = nn_cp["norm_params"]
nn_log_target = nn_cp.get("log_target", False)

dev_cont_nn = (dev_cont - nn_norm["means"]) / nn_norm["stds"]
dummy = np.zeros(len(dev_cat), dtype=np.float32)
loader = create_dataloader(dev_cat, dev_cont_nn, dummy, batch_size=16384, shuffle=False, num_workers=0, pin_memory=False)

nn_preds = []
with torch.no_grad():
    for pu, do, c, _ in loader:
        p = nn_model(pu, do, c)
        if nn_log_target:
            p = p.exp()
        nn_preds.append(p.cpu().numpy())
nn_preds = np.concatenate(nn_preds)
del nn_model, nn_cp; gc.collect()
print(f"NN MAE: {np.mean(np.abs(nn_preds - dev_targets)):.1f}s")

# --- LGBM predictions ---
lgbm_model = lgb.Booster(model_file="lgbm_model.txt")
X_lgbm = build_lgbm_features(dev_cat, dev_cont)
lgbm_preds = lgbm_model.predict(X_lgbm)
del lgbm_model, X_lgbm; gc.collect()
print(f"LGBM MAE: {np.mean(np.abs(lgbm_preds - dev_targets)):.1f}s")

# --- FT predictions ---
ft_cp = torch.load("ft_model.pt", map_location="cpu", weights_only=False)
ft_config = FTConfig(**ft_cp["model_config"])
ft_model = FTTransformer(ft_config)
ft_model.load_state_dict(ft_cp["model_state_dict"])
ft_model.eval()
ft_norm = ft_cp["norm_params"]

dev_cont_ft = (dev_cont - ft_norm["means"]) / ft_norm["stds"]
x_num_t = torch.from_numpy(dev_cont_ft).float()
x_cat_t = torch.from_numpy(dev_cat).long()

ft_preds = []
with torch.no_grad():
    for i in range(0, len(x_num_t), 16384):
        p = ft_model(x_num_t[i:i+16384], x_cat_t[i:i+16384])
        ft_preds.append(p.cpu().numpy())
ft_preds = np.concatenate(ft_preds)
del ft_model, ft_cp; gc.collect()
print(f"FT MAE: {np.mean(np.abs(ft_preds - dev_targets)):.1f}s")

# --- Grid search 3-model weights ---
print(f"\n{'alpha_nn':<10} {'alpha_lgbm':<12} {'alpha_ft':<10} {'MAE':>8} {'Bias':>8}")
print("-" * 55)

best_mae = float("inf")
best_weights = (0.5, 0.5, 0.0)

for a_nn in np.arange(0.0, 1.05, 0.1):
    for a_lgbm in np.arange(0.0, 1.05 - a_nn, 0.1):
        a_ft = round(1.0 - a_nn - a_lgbm, 2)
        if a_ft < -0.01:
            continue
        a_ft = max(a_ft, 0.0)
        ens = a_nn * nn_preds + a_lgbm * lgbm_preds + a_ft * ft_preds
        mae = float(np.mean(np.abs(ens - dev_targets)))
        bias = float(np.mean(ens - dev_targets))
        if mae < best_mae:
            best_mae = mae
            best_weights = (round(a_nn, 2), round(a_lgbm, 2), round(a_ft, 2))
            print(f"{a_nn:<10.2f} {a_lgbm:<12.2f} {a_ft:<10.2f} {mae:>8.1f} {bias:>+8.1f}  <-- best")

print(f"\nBest weights: NN={best_weights[0]}, LGBM={best_weights[1]}, FT={best_weights[2]}")
print(f"Best 3-model MAE: {best_mae:.1f}s")
print(f"2-model (NN+LGBM) MAE was: 254.0s")
print(f"Improvement: {best_mae - 254.0:+.1f}s")

---
## Results + Upload

In [ ]:
# 6. Check all model results
print("=" * 50)
print("MODEL SUMMARY")
print("=" * 50)

# NN
checkpoint = torch.load("model.pt", map_location="cpu", weights_only=False)
print(f"\nNN:")
print(f"  Dev MAE: {checkpoint['dev_mae']:.1f} s")
print(f"  Epoch: {checkpoint['epoch']}")
print(f"  Config: {checkpoint['model_config']}")

# LGBM
if os.path.exists("lgbm_model.txt"):
    lgbm = lgb.Booster(model_file="lgbm_model.txt")
    print(f"\nLGBM:")
    print(f"  Trees: {lgbm.num_trees()}")
    print(f"  Size: {os.path.getsize('lgbm_model.txt') / 1e6:.1f} MB")

# FT-Transformer
if os.path.exists("ft_model.pt"):
    ft_cp = torch.load("ft_model.pt", map_location="cpu", weights_only=False)
    print(f"\nFT-Transformer:")
    print(f"  Dev MAE: {ft_cp['dev_mae']:.1f} s")
    print(f"  Epoch: {ft_cp['epoch']}")
    print(f"  Config: {ft_cp['model_config']}")

In [ ]:
# 7. Upload all models to HF Hub
from huggingface_hub import HfApi

MODEL_REPO = "sarthakbiswas/eta-engine"
VERSION = "v5"  # <-- change this each run

# Get HF token
token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass

if token:
    api = HfApi(token=token)
    api.create_repo(repo_id=MODEL_REPO, exist_ok=True)

    # NN
    api.upload_file(path_or_fileobj="model.pt", path_in_repo=f"model_{VERSION}.pt", repo_id=MODEL_REPO)
    api.upload_file(path_or_fileobj="model.pt", path_in_repo="model.pt", repo_id=MODEL_REPO)
    print(f"Uploaded NN model_{VERSION}.pt + model.pt")

    # LGBM
    if os.path.exists("lgbm_model.txt"):
        api.upload_file(path_or_fileobj="lgbm_model.txt", path_in_repo="lgbm_model.txt", repo_id=MODEL_REPO)
        print("Uploaded lgbm_model.txt")

    # FT-Transformer
    if os.path.exists("ft_model.pt"):
        api.upload_file(path_or_fileobj="ft_model.pt", path_in_repo=f"ft_model_{VERSION}.pt", repo_id=MODEL_REPO)
        api.upload_file(path_or_fileobj="ft_model.pt", path_in_repo="ft_model.pt", repo_id=MODEL_REPO)
        print(f"Uploaded FT ft_model_{VERSION}.pt + ft_model.pt")

    print(f"https://huggingface.co/{MODEL_REPO}")
else:
    print("No HF_TOKEN found.")

In [ ]:
# 8. Archive MLflow runs to HF Hub
import tarfile

mlruns_tar = "mlruns.tar.gz"
with tarfile.open(mlruns_tar, "w:gz") as tar:
    tar.add("mlruns", arcname="mlruns")
print(f"Archived mlruns to {mlruns_tar}")

if token:
    api.upload_file(path_or_fileobj=mlruns_tar, path_in_repo="mlruns.tar.gz", repo_id=MODEL_REPO)
    print("MLflow runs uploaded")
else:
    print("No HF_TOKEN.")